# Error Handling

This notebook shows how to handle errors effectively when using the Biomedical Knowledge Lookup library.

## Common Error Types

In [ ]:
from knowledge_lookup.exceptions import (
    SourceUnavailableError,
    RateLimitError,
    APIError,
    ConfigurationError,
    NetworkError
)

# List of exception types
exceptions = [
    "SourceUnavailableError",
    "RateLimitError",
    "APIError",
    "ConfigurationError",
    "NetworkError"
]

print("Common exception types:")
for exc in exceptions:
    print(f"  - {exc}")

## Graceful Degradation

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup, LookupConfig
from knowledge_lookup.exceptions import SourceUnavailableError, RateLimitError

async def search_with_degradation(query, sources):
    """Search with graceful degradation when sources fail."""
    lookup = CentralKnowledgeLookup()
    
    all_results = []
    
    for source in sources:
        try:
            result = await lookup.search_concepts(
                query,
                sources=[source],
                limit=10
            )
            all_results.extend(result.concepts)
            print(f"{source}: {len(result.concepts)} results")
        except (SourceUnavailableError, RateLimitError) as e:
            print(f"{source} unavailable: {e}")
            # Continue with other sources
            continue
        except Exception as e:
            print(f"Unexpected error from {source}: {e}")
            continue
    
    await lookup.close()
    return all_results

# Run the search
results = asyncio.run(search_with_degradation(
    "diabetes",
    ["BioPortal", "UMLS", "ChEMBL"]
))
print(f"\nTotal results: {len(results)}")

## Retry with Fallback

In [ ]:
import asyncio
import random
from knowledge_lookup import CentralKnowledgeLookup
from knowledge_lookup.exceptions import RateLimitError

async def search_with_retry(lookup, query, sources, max_retries=3):
    """Search with automatic retry and fallback."""
    for attempt in range(max_retries):
        try:
            return await lookup.search_concepts(query, sources=sources)
        except RateLimitError as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt + random.uniform(0, 1)
                print(f"Rate limited, retrying in {wait_time:.2f}s...")
                await asyncio.sleep(wait_time)
            else:
                raise

async def retry_example():
    lookup = CentralKnowledgeLookup()
    
    try:
        result = await search_with_retry(
            lookup,
            "diabetes",
            sources=["BioPortal"],
            max_retries=5
        )
        print(f"Results: {len(result.concepts)}")
    except RateLimitError as e:
        print(f"Max retries exceeded. Please wait and try again.")
    finally:
        await lookup.close()

await retry_example()

## Health Checks

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup

async def check_source_health():
    """Check health of all sources before querying."""
    lookup = CentralKnowledgeLookup()
    
    healthy = []
    unhealthy = []
    
    for source in lookup.available_sources:
        try:
            available = await lookup.is_source_available(source)
            if available:
                healthy.append(source)
            else:
                unhealthy.append(source)
        except Exception as e:
            unhealthy.append(source)
            print(f"Error checking {source}: {e}")
    
    print(f"Healthy sources: {healthy}")
    print(f"Unhealthy sources: {unhealthy}")
    
    await lookup.close()

await check_source_health()

## Logging Errors

In [ ]:
import logging
import asyncio
from knowledge_lookup import CentralKnowledgeLookup

# Configure logging
logging.basicConfig(
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('biomedical_errors.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('biomedical_lookup')

async def search_with_logging(lookup, query):
    """Search with custom error logging."""
    try:
        result = await lookup.search_concepts(query)
        return result
    except Exception as e:
        logger.error(f"Search error for '{query}': {type(e).__name__}: {e}")
        raise

async def logging_example():
    lookup = CentralKnowledgeLookup()
    
    try:
        result = await search_with_logging(lookup, "diabetes")
        print(f"Results: {len(result.concepts)}")
    except Exception as e:
        print(f"Error logged. Check biomedical_errors.log")
    finally:
        await lookup.close()

await logging_example()

## Best Practices

In [ ]:
# 1. Always handle specific exception types
# 2. Implement retry logic for transient errors
# 3. Use graceful degradation for multiple sources
# 4. Perform health checks before queries
# 5. Log errors for debugging
# 6. Provide user-friendly error messages

# Example pattern:
# try:
#     result = await lookup.search_concepts(query)
# except RateLimitError as e:
#     print(f"Rate limit reached. Wait {e.retry_after}s and retry.")
# except SourceUnavailableError as e:
#     print(f"Source {e.source} is unavailable. Try another source.")

## Next Steps

- See `docs/guides/error_handling.md` for detailed error handling guide